## Transformación de los datos a DataFrame

Los logs originales de Apache se encuentran en formato texto plano, lo que requiere un proceso de transformación para estructurar la información.

Se ha utilizado Python junto con expresiones regulares para extraer los campos relevantes de cada línea del log y convertirlos en una estructura tabular (DataFrame).

Los pasos principales del proceso son:

1. Lectura del archivo línea por línea.
2. Limpieza básica de caracteres innecesarios (comillas y espacios).
3. Aplicación de una expresión regular para extraer los campos del log.
4. Conversión de los registros en un DataFrame de pandas.
5. Transformación de tipos de datos (fechas y valores numéricos).

Este proceso permite pasar de datos no estructurados a un formato analizable.

In [3]:
import re
import pandas as pd

data = []

with open("Datos/Originales/apache-logs.txt", "r", encoding="utf-8", errors="ignore") as f:
    for line in f:

        line = line.strip()

        if line.startswith("'") and line.endswith("'"):
            line = line[1:-1]

        if line.startswith('"') and line.endswith('"'):
            line = line[1:-1]

        line = line.replace("\\n", "").strip()

        data.append(line)

pattern = re.compile(
    r'(?P<ip>\S+) - - \[(?P<time>[^\]]+)\] '
    r'"(?P<method>\S+) (?P<url>\S+) (?P<protocol>[^"]+)" '
    r'(?P<status>\d{3}) (?P<size>\S+) '
    r'"(?P<referer>[^"]*)" "(?P<agent>[^"]*)"'
)

rows = []

for line in data:
    match = pattern.match(line)
    if match:
        rows.append(match.groupdict())

df = pd.DataFrame(rows)

month_map = {
    "Jan": "01", "Feb": "02", "Mar": "03", "Apr": "04",
    "May": "05", "Jun": "06", "Jul": "07", "Aug": "08",
    "Sep": "09", "Oct": "10", "Nov": "11", "Dec": "12",
}
ts = df["time"].astype(str)
for name, num in month_map.items():
    ts = ts.str.replace(f"/{name}/", f"/{num}/", regex=False)

df["time"] = pd.to_datetime(
    ts, format="%d/%m/%Y:%H:%M:%S %z", errors="coerce", utc=True
)

df["status"] = pd.to_numeric(df["status"], errors="coerce")
df["size"] = pd.to_numeric(df["size"], errors="coerce").fillna(0).astype(int)

# Análisis exploratorio inicial de los logs Apache

## 1. Descripción general del dataset
El conjunto de datos está compuesto por registros de acceso a un servidor web Apache correspondiente a una aplicación vulnerable (DVWA). Cada fila representa una petición HTTP realizada al servidor.

Las variables principales incluyen:
- Dirección IP del cliente
- Fecha y hora de la petición
- Método HTTP (GET, HEAD, etc.)
- URL solicitada
- Código de estado HTTP
- Tamaño de la respuesta
- Referer
- User-Agent del cliente

---

## 2. Dimensión del dataset
Se realiza una primera inspección del volumen de datos para entender la escala del análisis.

- Número de registros: elevado (miles de peticiones)
- Estructura: datos completamente tabulares tras el proceso de transformación

In [4]:
print("Filas:", len(df))
print("Columnas:", df.shape[1])

Filas: 13622
Columnas: 9


In [5]:
df["ip"].value_counts()

ip
192.168.4.164    7090
192.168.4.25     6532
Name: count, dtype: int64

In [6]:
df["status"].value_counts().sort_index()

status
200    5619
301      18
302     998
303    3407
400      11
403       3
404    2842
405      12
500     712
Name: count, dtype: int64

In [7]:
df["method"].value_counts()

method
POST          6689
GET           4826
HEAD          2038
OPTIONS         34
PROPFIND        34
NETSPARKER       1
Name: count, dtype: int64

In [8]:
df["url"].value_counts().head(15)

url
/index.php/component/search/                              4167
/DVWA/login.php                                           1535
/index.php                                                 579
/administrator/index.php                                   307
/DVWA/setup.php                                            301
/DVWA/vulnerabilities/exec/                                291
/index.php/component/users/?task=user.login                184
/index.php/component/users/?task=registration.register     114
/index.php/component/users/?task=reset.request              27
/index.php/component/users/?task=remind.remind              19
/DVWA/                                                      17
/DVWA/security.php                                          17
/DVWA/vulnerabilities/xss_s/                                16
/DVWA/vulnerabilities/brute/                                12
/DVWA/vulnerabilities/captcha/                              12
Name: count, dtype: int64

In [9]:
df["agent"].value_counts()

agent
Mozilla/5.0 (Windows NT 6.3; WOW64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/41.0.2272.16 Safari/537.36    7090
Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebKit/537.21 (KHTML, like Gecko) Chrome/41.0.2228.0 Safari/537.21     6530
Mozilla/4.0 (compatible; MSIE 8.0; Windows NT 6.1; Trident/4.0; w3af.sf.net)                                       2
Name: count, dtype: int64

In [10]:
df["time"].dt.floor("min").value_counts().sort_index().head(20)

time
2022-12-22 13:11:00+00:00      2
2022-12-22 13:18:00+00:00    342
2022-12-22 13:19:00+00:00    338
2022-12-22 13:20:00+00:00    372
2022-12-22 13:21:00+00:00    363
2022-12-22 13:22:00+00:00    363
2022-12-22 13:23:00+00:00    340
2022-12-22 13:24:00+00:00    268
2022-12-22 13:25:00+00:00    323
2022-12-22 13:26:00+00:00    410
2022-12-22 13:27:00+00:00    399
2022-12-22 13:28:00+00:00    414
2022-12-22 13:29:00+00:00    317
2022-12-22 13:30:00+00:00    298
2022-12-22 13:31:00+00:00    259
2022-12-22 13:32:00+00:00    253
2022-12-22 13:33:00+00:00    190
2022-12-22 13:34:00+00:00    194
2022-12-22 13:35:00+00:00    181
2022-12-22 13:36:00+00:00    206
Name: count, dtype: int64

# Detección de anomalías en los logs Apache

## 1. Objetivo del análisis
El objetivo de esta fase es identificar comportamientos anómalos dentro del tráfico web registrado, que puedan indicar intentos de ataque, escaneo de vulnerabilidades o actividad automatizada.

---

## 2. Identificación de patrones sospechosos
Durante el análisis de los registros se han detectado posibles indicios de actividad anómala:

- Accesos repetidos desde la misma dirección IP en cortos intervalos de tiempo.
- Solicitudes a rutas sensibles o inusuales como:
  - `/DVWA/.git/`
  - `/DVWA/.svn/`
  - `/DVWA/config`
- Uso recurrente de métodos HTTP como `HEAD` para reconocimiento del servidor.
- Presencia de múltiples códigos de error HTTP (especialmente 404), lo que puede indicar enumeración de recursos inexistentes.

In [15]:
# URLs sospechosas
suspicious_patterns = [
    r"\.git", r"\.svn", r"\.env",
    r"wp-admin", r"wp-login",
    r"admin", r"login",
    r"config", r"backup",
    r"\.sql", r"\.db",
    r"shell", r"cmd",
    r"\.\./"  
]

pattern = "|".join(suspicious_patterns)

df["is_suspicious"] = df["url"].str.contains(pattern, case=False, na=False)

suspicious = df[df["is_suspicious"]]

print("\nAnomalías detectadas:", len(suspicious))

print("\nTop URLs sospechosas:")
print(suspicious["url"].value_counts().head(10))

print("\nIPs con más actividad sospechosa:")
print(suspicious["ip"].value_counts().head(10))

print("\nMétodos usados en ataques:")
print(suspicious["method"].value_counts())



Anomalías detectadas: 3056

Top URLs sospechosas:
url
/DVWA/login.php                                              1535
/administrator/index.php                                      307
/index.php/component/users/?task=user.login                   184
/DVWA/dvwa/css/login.css                                        4
/DVWA/login.php?nsextt=%0ans%3anetsparker056650%3dvuln          3
/DVWA/login.php?nsextt=%0d%0ans%3anetsparker056650%3dvuln       3
/DVWA/.git/config                                               2
/DVWA/.svn/all-wcprops                                          2
/DVWA/.svn/wc.db                                                2
/DVWA/config.inc                                                2
Name: count, dtype: int64

IPs con más actividad sospechosa:
ip
192.168.4.164    2339
192.168.4.25      717
Name: count, dtype: int64

Métodos usados en ataques:
method
POST    1275
GET      959
HEAD     822
Name: count, dtype: int64


## Conclusiones anomalias

- Se detecta un volumen elevado de peticiones a rutas sensibles como `/DVWA/login.php` y paneles de administración, lo que indica intentos de acceso no autorizado.

- El alto número de solicitudes mediante el método `POST` sugiere un posible **ataque de fuerza bruta** contra formularios de login.

- La presencia de accesos a rutas como `.git`, `.svn` o archivos de configuración evidencia un **escaneo de vulnerabilidades automatizado**.

- El uso frecuente del método `HEAD` y la repetición de peticiones refuerzan la hipótesis de tráfico generado por bots.

- En conjunto, los patrones observados corresponden a un **ataque automatizado que combina escaneo y explotación básica del sistema**.

In [16]:
# Errores HTTP
errors = df[df["status"] >= 400]

print("Errores HTTP:", len(errors))

print("\nDistribución de errores:")
print(errors["status"].value_counts())

print("\nIPs con más errores:")
print(errors["ip"].value_counts().head(10))

Errores HTTP: 3580

Distribución de errores:
status
404    2842
500     712
405      12
400      11
403       3
Name: count, dtype: int64

IPs con más errores:
ip
192.168.4.164    2605
192.168.4.25      975
Name: count, dtype: int64


## Conclusiones HTTP

- Se han registrado un total de **3580 errores HTTP**, lo que indica una actividad anómala significativa en el servidor.

- La mayoría de los errores corresponden a códigos **404 (Not Found)** con 2842 ocurrencias, lo que sugiere intentos de acceso a rutas inexistentes, típicos de escaneos automatizados.

- Se detectan **712 errores 500 (Internal Server Error)**, lo que podría indicar intentos de explotación que provocan fallos en el servidor.

- Otros errores como **405, 400 y 403** aparecen en menor medida, pero refuerzan la presencia de peticiones mal formadas o no autorizadas.

- En conjunto, el patrón de errores confirma un comportamiento no legítimo orientado a descubrir recursos y posibles fallos del sistema.

- El reducido número de direcciones IP se debe probablemente a que los datos provienen de un entorno controlado o de laboratorio, donde un número limitado de máquinas genera tráfico. Además, los ataques automatizados no requieren múltiples orígenes, ya que una sola IP puede generar un alto volumen de peticiones maliciosas.

In [18]:
risk = df.groupby("ip").agg({
    "is_suspicious": "sum",
    "status": lambda x: (x >= 400).sum(),
    "url": "count"
})

risk.columns = ["suspicious_count", "error_count", "total_requests"]

# ratio de error
risk["error_ratio"] = risk["error_count"] / risk["total_requests"]

# score simple
risk["risk_score"] = (
    risk["suspicious_count"] * 2 +
    risk["error_count"] +
    risk["error_ratio"] * 10
)

print("\nIPs más peligrosas:")
print(risk.sort_values("risk_score", ascending=False).head(10))


IPs más peligrosas:
               suspicious_count  error_count  total_requests  error_ratio  \
ip                                                                          
192.168.4.164              2339         2605            7090     0.367419   
192.168.4.25                717          975            6532     0.149265   

                risk_score  
ip                          
192.168.4.164  7286.674189  
192.168.4.25   2410.492652  


In [20]:
risk = df.groupby("ip").agg({
    "method": lambda x: (x == "POST").sum(),
    "status": lambda x: (x >= 400).sum(),
    "is_suspicious": "sum",
    "url": "count"
})

risk.columns = ["post_requests", "errors", "suspicious", "total"]

risk["attack_type"] = "normal"

risk.loc[risk["post_requests"] > 100, "attack_type"] = "brute_force"
risk.loc[risk["suspicious"] > 50, "attack_type"] = "scanner"
risk.loc[(risk["post_requests"] > 50) & (risk["suspicious"] > 50), "attack_type"] = "mixed_attack"

print(risk.sort_values("total", ascending=False))

               post_requests  errors  suspicious  total   attack_type
ip                                                                   
192.168.4.164           1190    2605        2339   7090  mixed_attack
192.168.4.25            5499     975         717   6532  mixed_attack


In [22]:
df["minute"] = df["time"].dt.floor("min")

traffic_spikes = (
    df.groupby("minute")
    .size()
    .sort_values(ascending=False)
    .head(10)
)

error_spikes = (
    df[df["status"] >= 400]
    .groupby("minute")
    .size()
    .sort_values(ascending=False)
    .head(10)
)

anomaly_spikes = (
    df[df["is_suspicious"] == True]
    .groupby("minute")
    .size()
    .sort_values(ascending=False)
    .head(10)
)

summary = pd.DataFrame({
    "traffic": df.groupby("minute").size(),
    "errors": df[df["status"] >= 400].groupby("minute").size(),
    "anomalies": df[df["is_suspicious"] == True].groupby("minute").size()
}).fillna(0)

summary["risk_score"] = (
    summary["errors"] * 2 +
    summary["anomalies"] * 3 +
    summary["traffic"] * 0.5
)

risk_minutes = summary.sort_values("risk_score", ascending=False).head(10)

traffic_spikes
error_spikes
anomaly_spikes
risk_minutes

,traffic,errors,anomalies,risk_score
minute,,,,
2023-12-22 12:19:00+00:00,2833,2279.0,817.0,8425.5
2023-12-22 12:20:00+00:00,2567,214.0,1438.0,6025.5
2023-12-22 12:21:00+00:00,1370,81.0,82.0,1093.0
2022-12-22 13:29:00+00:00,317,73.0,103.0,613.5
2022-12-22 13:28:00+00:00,414,134.0,40.0,595.0
2022-12-22 13:30:00+00:00,298,39.0,102.0,533.0
2022-12-22 13:19:00+00:00,338,146.0,21.0,524.0
2022-12-22 13:18:00+00:00,342,90.0,39.0,468.0
2022-12-22 13:31:00+00:00,259,80.0,53.0,448.5
